# Using Gemini to help me with my Photography business

I own a photography studio. There many tasks during my day that take a lot of time and effort.
I can use Google's Gemini AI to reduce some of this effort.

## Setup

First, install ChromaDB and the Gemini API Python SDK.

Also install pypdfplumber for document parsing

In [1]:
!pip uninstall -qqy jupyterlab kfp  # Remove unused conflicting packages
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"

In [2]:
from google import genai
from google.genai import types

from IPython.display import Markdown

genai.__version__

'1.7.0'

In [3]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

In [4]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 56.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 74.9 MB/s eta 0:00:00:00:01


## 1. Help me go through a long document about my camera
#### I own a Nikon Z7II and it is very helpful to be able look through it's document for a specific setting
#### Set up a RAG framework for Gemini to go through this document and answer my specific question

First, let's convert the pdf to text documents

In [5]:
import pdfplumber

def extract_text_from_pdf(pdf_file):
    documents = []
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            documents.append(page.extract_text())
    return documents

In [6]:
documents = extract_text_from_pdf("/kaggle/input/nikon-z72-settings/Z7IIZ6II_TG_Setting.pdf")

Use Gemini Embeddings and store this document in chroma DB

In [7]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry

from google.genai import types


# Define a helper to retry when per-minute quota is reached.
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})


class GeminiEmbeddingFunction(EmbeddingFunction):
    # Specify whether to generate embeddings for documents, or queries
    document_mode = True

    @retry.Retry(predicate=is_retriable)
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = "retrieval_document"
        else:
            embedding_task = "retrieval_query"

        response = client.models.embed_content(
            model="models/text-embedding-004",
            contents=input,
            config=types.EmbedContentConfig(
                task_type=embedding_task,
            ),
        )
        return [e.values for e in response.embeddings]

In [8]:
import chromadb

client = genai.Client(api_key=GOOGLE_API_KEY)

In [9]:
DB_NAME = "NikonZ7IIdb"

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

db.add(documents=documents, ids=[str(i) for i in range(len(documents))])

In [10]:
db.count()

72

Now, let's ask this RAG framework about time-lapse photography

In [11]:
# Switch to query mode when generating embeddings.
embed_fn.document_mode = False

# Search the Chroma DB using the specified query.
query = "How do I prepare and set for time-lapse recording?"

result = db.query(query_texts=[query], n_results=1)
[all_passages] = result["documents"]

Markdown(all_passages[0])

RReeaaddyy aa TTrriippoodd
Use a tripod for time-lapse recording. Be sure to choose
one that won’t tip or blow over.
●Before Shooting
PPoossiittiioonn tthhee CCaammeerraa
Choose a location, position the tripod, and secure the cam-
era. Check that the camera is steady, the power cable un-
obstructed, and the power source within reach.
CChheecckk FFrraammiinngg aanndd FFooccuuss
• Take a test shot and check framing and focus.
• Time-lapse footage shot with [FX (36 × 24)] selected for
[Choose image area] in the photo shooting menu will be
recorded at an aspect ratio of 16∶9. We recommend that
you switch briefly to [16:9 (36 × 20)] when framing shots.
Quick Image‑Area Selection
Assigning [Choose image area] to the Fn1 or Fn2 button
using Custom Setting f2 [Custom controls] lets you quickly
switch image areas by holding the button and rotating a
command dial.
DDeeaalliinngg wwiitthh CCoonnddeennssaattiioonn
We recommend using a third-party lens heater in settings
where condensation is likely.
59

In [12]:
query_oneline = query.replace("\n", " ")

# This prompt is where you can specify any guidance on tone, or what topics the model should stick to, or avoid.
prompt = f"""You are a helpful and informative bot that answers questions using text from the reference passage included below. 
Be sure to respond in a complete sentence, being comprehensive, including all relevant background information. 
You are talking to a technical audience, so be sure to break down complicated concepts and 
strike a friendly and converstional tone. If the passage is irrelevant to the answer, you may ignore it.

QUESTION: {query_oneline}
"""

# Add the retrieved documents to the prompt.
for passage in all_passages:
    passage_oneline = passage.replace("\n", " ")
    prompt += f"PASSAGE: {passage_oneline}\n"

In [13]:
answer = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt)

Markdown(answer.text)

To prepare for time-lapse recording, you'll want to use a tripod that won't tip or blow over, position the camera, and secure it to the tripod while ensuring the power cable is unobstructed and the power source is within reach. It's also important to take a test shot to check the framing and focus, and to consider using a third-party lens heater in environments where condensation is likely to occur.


Awesome! That was helpful :)

 ## 2. Help me research and reach out to my clients
#### I am always looking to expand my client base as well as increase business with current clients. Reaching out to these clients takes up a lot of time to research who to reach out to and draft cold emails.
#### I will use Gemini to collect data on such clients
#### I will store this research in a sql db that I can later filter potential clients according to my needs and reach out to them

In [20]:
import json
import sqlite3

In [15]:
import typing_extensions as typing

class ClientList(typing.TypedDict):
    name_of_company: str
    headquarters: str
    top_5_products: list[str]
    revenue_USD: int
    marketing_manager: str


In [16]:
query = """You are a helpful AI assistant helping me build a client base.
Search for the top 20 food product companies in the US and collect the following information for all of them in the form of a json-
name_of_company: name of the company
headquarters: where is the headquarters of the company
top_5_products: list of top 5 food products by the company by volume
revenue_USD: yearly revenue of the company in US dollars
marketing_manager: the media marketing manager, or equivalent, that I can reach out to to pitch my services

present this information in json format in the following way -
[{   'name_of_company': 'Pepsi'
    'headquarters': 'Purchase, New York'
    top_5_products: 'Pepsi,Mountain Dew,Lay's Potato Chips,Gatorade,Tropicana'
    revenue_USD: 91,470,000,000
    marketing_manager: 'Greg Lyons, Chief Marketing Officer, PepsiCo Beverages North America'},
{   'name_of_company': 'Tyson Foods'
    'headquarters': 'Springdale, Arkansas'
    top_5_products: 'Coffee-Mate,Purina Pet Food,Lean Cuisine,Häagen-Dazs,Kit Kat'
    revenue_USD: 52,880,000,000
    marketing_manager: 'Melanie Boulden, Chief Growth Officer'},
    ...]
"""

In [17]:
response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=query,
    config=types.GenerateContentConfig(
        temperature = 0.1
    )
)

In [18]:
client_list_text = """" """ + response.text + """ """
client_list_text = client_list_text.replace("```json","")
client_list_text = client_list_text.replace("\n","")
client_list_text = client_list_text.replace('`',"")
client_list_text = client_list_text.replace('", "',",")
client_list_text = client_list_text.replace('["','"')
client_list_text = client_list_text.replace('"]','"')
client_list_text = client_list_text[1:]

In [21]:
client_list_json = json.loads(client_list_text)

In [22]:
client_list_json

[{'name_of_company': 'PepsiCo',
  'headquarters': 'Purchase, New York',
  'top_5_products': "Pepsi,Mountain Dew,Lay's Potato Chips,Gatorade,Tropicana",
  'revenue_USD': 91470000000,
  'marketing_manager': 'Greg Lyons, Chief Marketing Officer, PepsiCo Beverages North America'},
 {'name_of_company': 'Tyson Foods',
  'headquarters': 'Springdale, Arkansas',
  'top_5_products': 'Chicken,Beef,Pork,Prepared Foods,Jimmy Dean',
  'revenue_USD': 52880000000,
  'marketing_manager': 'Melanie Boulden, Chief Growth Officer'},
 {'name_of_company': 'Nestlé USA',
  'headquarters': 'Arlington, Virginia',
  'top_5_products': 'Coffee-Mate,Purina Pet Food,Lean Cuisine,Häagen-Dazs,Kit Kat',
  'revenue_USD': 32000000000,
  'marketing_manager': 'Alicia Enciso, Chief Marketing Officer, Nestlé USA'},
 {'name_of_company': 'Kraft Heinz',
  'headquarters': 'Chicago, Illinois',
  'top_5_products': 'Kraft Macaroni & Cheese,Heinz Ketchup,Oscar Mayer,Philadelphia Cream Cheese,Lunchables',
  'revenue_USD': 26500000000,

In [23]:
db_file = "clientlist.db"
con = sqlite3.connect(db_file)
cur = con.cursor()

# ... perform database operations ...

# Close the existing connection
con.close()

In [24]:
# Open the file containing the SQL database.
with sqlite3.connect("clientlist.db") as conn:
    
    # Create the table if it doesn't exist.
    conn.execute(
        """CREATE TABLE IF NOT EXISTS client_list1(
                name_of_company varchar(100),
                headquarters varchar(100),
                top_5_products varchar(100),
                revenue_USD int,
                marketing_manager varchar(200)
            );"""
        )

    # Insert each entry from json into the table.
    keys = ["name_of_company", "headquarters", "top_5_products", "revenue_USD","marketing_manager"]
    for entry in client_list_json:

        # This will make sure that each key will default to None
        # if the key doesn't exist in the json entry.
        values = [entry.get(key, None) for key in keys]

        # Execute the command and replace '?' with the each value
        # in 'values'. DO NOT build a string and replace manually.
        # the sqlite3 library will handle non safe strings by doing this.
        cmd = """INSERT INTO client_list1 VALUES(
                    ?,
                    ?,
                    ?,
                    ?,
                    ?
                );"""
        conn.execute(cmd, values)

    conn.commit()

In [25]:
def execute_query(sql: str) -> list[list[str]]:
    """Execute an SQL statement, returning the results."""
    print(f' - DB CALL: execute_query({sql})')

    cursor = conn.cursor()

    cursor.execute(sql)
    return cursor.fetchall()


In [26]:
db_tools = [execute_query]

instruction = """You are a helpful chatbot that can interact with an SQL database
for a list of clients. You will take the users questions and turn them into SQL
queries using the tools available. Once you have the information you need, you will
answer the user's question using the data returned.

Use execute_query to issue an SQL SELECT query."""


# Start a chat with automatic function calling enabled.
chat = client.chats.create(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
        system_instruction=instruction,
        tools=db_tools,
    ),
)

### I have a brilliant photography idea for Ketchup. Let me reach out to someone who would be interested in that!

In [30]:
resp = chat.send_message("What is the name ofcompany from client_list1 which has ketchup in the top 5 products?")
print(f"\n{resp.text}")

 - DB CALL: execute_query(SELECT name_of_company FROM client_list1 WHERE top_5_products LIKE '%Ketchup%')

The name of the company from client_list1 which has ketchup in the top 5 products is Kraft Heinz.



In [31]:
resp = chat.send_message("Who is their marketing manager?")
print(f"\n{resp.text}")

 - DB CALL: execute_query(SELECT marketing_manager FROM client_list1 WHERE name_of_company = 'Kraft Heinz')

The marketing manager for Kraft Heinz is Diana Frost, Chief Growth Officer, North America Zone.



### Great! that was helpful, Now let's draft an email to send to them

In [32]:
resp = chat.send_message("""Draft an email to them, pitching my photography studio named The Still Studio. 
Make it so that they know that we specialize in food photography providing stunning visuals. Mention my website at the end - thestillstudio.myportfolio.com""")
print(f"\n{resp.text}")


Subject: Elevate Your Brand with Stunning Food Photography - The Still Studio

Dear Diana Frost,

My name is [Your Name] and I'm reaching out from The Still Studio, a photography studio specializing in creating visually stunning food photography. We've been consistently impressed by Kraft Heinz's innovative products and engaging marketing campaigns. We believe that high-quality visuals are essential for showcasing the appeal of your food products in today's competitive market.

At The Still Studio, we understand the nuances of food photography. We know how to capture the textures, colors, and overall deliciousness of food to entice consumers and elevate your brand's image. We work closely with our clients to understand their brand identity and marketing objectives, ensuring that our photography aligns perfectly with their vision.

We're confident that The Still Studio can provide you with the kind of captivating imagery that will make your products stand out on shelves and online.

Pl

### Awesome!